In [29]:
from datetime import date, datetime, timedelta

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_ai_model.validation import flights_df_comparison
import polars as pl
DAY_MARGIN = timedelta(minutes=15)

test_date = date(2026,3,1)
df_gold = get_bts_flights_for_day(test_date)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = (pl.col("landing_time").dt.date() == test_date) & (pl.col("takeoff_time").dt.date() == test_date)
min_time = datetime(test_date.year, test_date.month, test_date.day) + DAY_MARGIN
max_time = datetime(test_date.year, test_date.month, test_date.day) + timedelta(days=1) - DAY_MARGIN
expr2 = (pl.col("takeoff_time") > min_time) & (pl.col("landing_time") < max_time)
expr = expr & expr2
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))
df,stats = flights_df_comparison(df_gold, df_test)

226 overlapping flights found for 2026-03-01

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 4385
Predicted flights: 13996  |  Gold flights: 15515
Correctly matched: 13814  |  Predicted unmatched: 182  |  Gold flights missed: 1701
Precision: 98.70%  |  Recall: 89.04%  |  F1: 93.62%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 13774/13783 matched (99.93%)
landing_airport_ident: 13735/13737 matched (99.99%)
overall: 27509/27520 matched (99.96%)


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_7891/1884531950.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))


In [30]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "takeoff_airport_ident_df2", "landing_time_df2","landing_airport_ident_df2", "match_status"]
df = df.select(COLUMNS)

In [31]:
df.filter(pl.col("match_status") == "df2_only")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a011ac""",null,null,null,null,2026-03-01 13:52:12.921,"""KPHL""",2026-03-01 16:07:24.395,null,"""df2_only"""
"""a011ac""",null,null,null,null,2026-03-01 17:52:09.715,"""MYNN""",2026-03-01 20:09:12.347,"""KPHL""","""df2_only"""
"""a03618""",null,null,null,null,2026-03-01 21:40:40.728,"""KGEG""",2026-03-01 23:21:13.790,"""KDEN""","""df2_only"""
"""a037b2""",null,null,null,null,2026-03-01 18:05:58.947,"""KSLC""",2026-03-01 20:02:58.129,"""CYVR""","""df2_only"""
"""a037b2""",null,null,null,null,2026-03-01 21:37:27.968,"""CYVR""",2026-03-01 23:25:13.533,"""KSLC""","""df2_only"""
…,…,…,…,…,…,…,…,…,…
"""adf068""",null,null,null,null,2026-03-01 16:42:58.478,"""KDFW""",2026-03-01 18:52:13.432,"""MMUN""","""df2_only"""
"""adf068""",null,null,null,null,2026-03-01 20:36:57.231,"""MMUN""",2026-03-01 22:58:43.859,"""KDFW""","""df2_only"""
"""adf128""",null,null,null,null,2026-03-01 14:33:28.182,"""KBOS""",2026-03-01 18:56:56.507,"""TNCA""","""df2_only"""


In [33]:
df.filter(pl.col("icao") == "a011ac").sort("takeoff_time")

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str
"""a011ac""",null,null,null,null,2026-03-01 13:52:12.921,"""KPHL""",2026-03-01 16:07:24.395,null,"""df2_only"""
"""a011ac""",null,null,null,null,2026-03-01 17:52:09.715,"""MYNN""",2026-03-01 20:09:12.347,"""KPHL""","""df2_only"""
"""a011ac""",2026-03-01 11:00:00,"""KBOS""",2026-03-01 12:03:00,"""KPHL""",2026-03-01 10:59:58.271,"""KBOS""",2026-03-01 12:03:13.434,"""KPHL""","""matched"""
"""a011ac""",2026-03-01 21:54:00,"""KPHL""",2026-03-01 22:41:00,"""KBOS""",2026-03-01 21:53:55.926,"""KPHL""",2026-03-01 22:41:28.600,"""KBOS""","""matched"""
